# LDM experiments: train, inference, and release

**How to use this notebook:** Run section 1 and mount Drive in section 2. To start new models, list the HF data, select a collection to load, define `RUN_CONFIGS`, prepare data, and run sections 3â€“5. To work with a saved model, set `SELECT_RUN_ID` once in section 6, then run only the actions you want in sections 7â€“13. Snapshot, inference, analysis, and release each load what they need from that selected run. Checkpoints and reports live on Drive; temporary data caches and snapshots live in `/content/ldm`.


## 1. Load the GitHub code

**Clone/update cell:** Reads `GITHUB_TOKEN` from Colab Secrets, clones or pulls `appodoom/ldm` into `/content/ldm`, and prints the exact code commit. It does not start training. The next cell installs the package and runs its contract tests.


In [ ]:
import base64
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

project = Path("/content/ldm")
token = userdata.get("GITHUB_TOKEN")
assert token, "Add GITHUB_TOKEN to Colab Secrets and allow notebook access"
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
})
if (project / ".git").exists():
    subprocess.run(["git", "-C", str(project), "pull", "--ff-only", "origin", "main"],
                   env=git_env, check=True)
elif project.exists():
    raise RuntimeError("/content/ldm exists but is not the Git checkout")
else:
    subprocess.run(["git", "clone", "https://github.com/appodoom/ldm.git", str(project)],
                   env=git_env, check=True)
code_commit = subprocess.check_output(
    ["git", "-C", str(project), "rev-parse", "HEAD"], text=True).strip()
print("Code commit:", code_commit)


### Install and check the package

The next cell installs the cloned code into this Colab runtime and runs the repository tests. It stops if those tests fail.


In [ ]:
os.chdir(project)
print("Python:", sys.version.split()[0])
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[audio,analysis]"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               check=True)


## 2. Inspect and load HF data; mount Drive

**List cell:** Signs in to Hugging Face and lists the folders in `Appodoom/ldm-data`, plus five-line `.derbake` collections available for training. `DATA_REVISION` defaults to `main`; the cell resolves it to an exact commit so later cells use the same data. Set `SHOW_REPO_FILES = True` to print every file. The following cell selects and downloads one collection. The Drive cell mounts `MyDrive/LDM_FINAL` so runs survive a Colab reset.


In [ ]:
from collections import Counter
from huggingface_hub import HfApi, notebook_login

DATA_REPO_ID = 'Appodoom/ldm-data'
DATA_REVISION = 'main'  # Set a commit SHA here to inspect an older version.
SHOW_REPO_FILES = False
notebook_login()
data_api = HfApi()
DATA_COMMIT = data_api.repo_info(DATA_REPO_ID, repo_type='dataset', revision=DATA_REVISION).sha
DATA_FILES = sorted(data_api.list_repo_files(DATA_REPO_ID, repo_type='dataset', revision=DATA_COMMIT))
DATA_COLLECTIONS = {}
for repo_path in DATA_FILES:
    parts = repo_path.split('/')
    if len(parts) >= 3 and parts[-2] in {'normal', 'decorated'} and parts[-1].endswith('.derbake'):
        collection = '/'.join(parts[:-2])
        DATA_COLLECTIONS.setdefault(collection, {'normal': [], 'decorated': []})[parts[-2]].append(repo_path)
print('Dataset:', DATA_REPO_ID, '| Commit:', DATA_COMMIT)
print('Top-level contents:')
for folder, count in sorted(Counter(path.split('/')[0] if '/' in path else '(root files)' for path in DATA_FILES).items()):
    print(f'  {folder}: {count} files')
print('Five-line collections available for training:')
for collection, groups in sorted(DATA_COLLECTIONS.items()):
    if groups['normal'] or groups['decorated']:
        print(f"  {collection}: {len(groups['normal'])} normal, {len(groups['decorated'])} decorated")
if SHOW_REPO_FILES:
    print('All repository files:')
    print('\n'.join(DATA_FILES))


### Select and download a data collection

Select a collection for each source in `SELECT_DATA_COLLECTIONS`, or use `None` to skip a source. Normal and decorated sources are independent: they can use different collections, filenames, and cycle counts. The cell downloads the selected files and verifies each inventory separately. Each run chooses one source through `DataConfig.decoration`. Rerun preparation after changing a selection.


In [ ]:
from huggingface_hub import snapshot_download
from ldm.experiment_config import DecorationMode

# Each source is independent. Use different collections, or None to skip a source.
SELECT_DATA_COLLECTIONS = {
    DecorationMode.WITHOUT_DECORATION: 'five-line',
    DecorationMode.WITH_DECORATION: 'five-line',
}

def current_data_selection():
    return (DATA_REPO_ID, DATA_COMMIT, tuple(sorted(
        (mode.value, collection) for mode, collection in SELECT_DATA_COLLECTIONS.items()
        if collection is not None)))

requested_groups = {}
for mode, collection in SELECT_DATA_COLLECTIONS.items():
    if not isinstance(mode, DecorationMode):
        raise TypeError('Use DecorationMode keys in SELECT_DATA_COLLECTIONS')
    if collection is None:
        continue
    files = DATA_COLLECTIONS.get(collection, {}).get(mode.value, [])
    if not files:
        raise ValueError(f'No {mode.value} files in collection {collection!r}; choose a listed source or None')
    requested_groups[mode] = (collection, files)
if not requested_groups:
    raise ValueError('Select at least one data source')

data_root = Path(snapshot_download(repo_id=DATA_REPO_ID, repo_type='dataset',
    revision=DATA_COMMIT,
    allow_patterns=[file for _, files in requested_groups.values() for file in files],
    local_dir=project / '.derbake' / DATA_COMMIT))
DATA_SOURCES = {}
for mode, (collection, files) in requested_groups.items():
    source = data_root / collection / mode.value
    expected_names = {path.rsplit('/', 1)[-1] for path in files}
    if {path.name for path in source.glob('*.derbake')} != expected_names:
        raise RuntimeError(f'Downloaded {mode.value} files differ from the selected HF list')
    DATA_SOURCES[mode] = source
    print(mode.name, '| Collection:', collection, '| Files:', len(files), '| Source:', source)
SELECTED_DATA_SOURCE = current_data_selection()
print('Dataset commit:', DATA_COMMIT)


### Mount Google Drive

The next cell sets the durable root for new runs, inference, plots, and release records. `/content` data can disappear when the runtime ends.


In [ ]:
from google.colab import drive
import torch, json
drive.mount('/content/drive')
storage_root = Path('/content/drive/MyDrive/LDM_FINAL')
storage_root.mkdir(parents=True, exist_ok=True)
runs_root = storage_root / 'runs'
runs_root.mkdir(parents=True, exist_ok=True)
cache_root = project / 'datasets'
print('Runs:', runs_root, '| Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## 3. Define the run list

Keep every experiment in the single `RUN_CONFIGS` list below. Add another `RunConfig(...)` entry whenever you want another combination of data, model, and training settings. `DataConfig.variant` chooses `with_subd` or `no_subd`; `DataConfig.decoration` independently chooses `WITHOUT_DECORATION` or `WITH_DECORATION`. These give four combinations in the same run list. Each run uses only its selected source; sources are never paired or weighted. The preparation and training cells process every entry in list order. This cell validates settings and checks optional custom `name` values for duplicates. Without a name, the run gets the old automatic timestamp/settings/hash name. Nothing trains here.


In [ ]:
from ldm.experiment_config import (
    Activation, AttentionLayout, DatasetVariant, DecorationMode, PositionEncoding,
    DataConfig, ModelConfig, TrainConfig, EvalConfig, RunConfig, StructuredStreams,
)
from ldm.experiment_workflow import check_run_name

RUN_CONFIGS = [
    RunConfig(
        # name='my_run',  # Optional custom run name.
        data=DataConfig(
            variant=DatasetVariant.WITH_SUBD,
            context_size=1024,
            stride=512,
            max_beats_per_cycle=16,
            decoration=DecorationMode.WITHOUT_DECORATION,
        ),
        model=ModelConfig(
            layers=4,
            heads=4,
            embeddings=256,
            mlp_ratio=4,
            dropout=0.1,
            attention=AttentionLayout.ALTERNATING_GLOBAL_STRUCTURED,
            activation=Activation.TANH,
            position=PositionEncoding.ROPE,
            rope_theta=10000.0,
            rope_fraction=1.0,
            use_type_embeddings=True,
            concat_type_embeddings=False,
            streams=StructuredStreams(
                inside_beat=True,
                beat_memory=True,
                cycle_memory=True,
            ),
        ),
        train=TrainConfig(
            epochs=10,
            batch_size=4,
            learning_rate=0.0003,
            weight_decay=0.01,
            grad_clip=1.0,
            amp=True,  # FP16 autocast and gradient scaling on CUDA.
            allow_tf32=True,  # Faster float32 CUDA matrix multiplication.
            num_workers=2,  # CPU workers preparing training batches; 0 disables workers.
            pin_memory=True,
            persistent_workers=True,
            prefetch_factor=2,
            save_every_steps=200,
            log_every_steps=50,
            early_stopping=False,
        ),
        evaluation=EvalConfig(
            batch_size=4,
            every_steps=200,
            save_plots=True,
        ),
    ),
    # Add more RunConfig(...) entries here; each chooses subdivision and decoration in DataConfig.
]
assert RUN_CONFIGS, 'Add at least one RunConfig'
for index, run_config in enumerate(RUN_CONFIGS):
    print(f'Config {index}: {check_run_name(run_config, runs_root, RUN_CONFIGS)}')
    print(json.dumps(run_config.to_dict(), indent=2))
    print('Config hash:', run_config.fingerprint)


## 4. Prepare or reuse the dataset

This cell loops over every `RunConfig` and reads its data settings and the selected HF collection. It splits complete cycles **within each file** about 70/20/10, uses only the selected normal or decorated source, and creates model arrays under `/content/ldm/datasets/<variant>-<decoration>-<recipe-hash>`. It reuses that cache only when the full recipe and source bytes match. The cache is temporary and can be rebuilt.

Reuse verifies the SHA-256, shape, and dtype of every prepared array, including all routing arrays. The preparation version and readable normalization/splitting rules are saved in the manifest. Changing the preparation contract creates a new cache.


In [ ]:
from ldm.experiment_workflow import prepare_or_load_dataset
assert all(name in globals() for name in ('SELECTED_DATA_SOURCE', 'DATA_SOURCES', 'current_data_selection')), (
    'List and select HF data before preparing datasets')
assert globals().get('SELECTED_DATA_SOURCE') == current_data_selection(), (
    'Run the select-and-download cell after listing or changing HF data')
PREPARED_CONFIGS = tuple(RUN_CONFIGS)
PREPARED_SOURCE = SELECTED_DATA_SOURCE
PREPARED_DATASETS = []
for index, run_config in enumerate(RUN_CONFIGS):
    print(f'Preparing config {index + 1}/{len(RUN_CONFIGS)}: {run_config.data.variant.value}')
    mode = run_config.data.decoration
    if mode not in DATA_SOURCES:
        raise ValueError(f'Load the {mode.value} source before preparing this config')
    prepared = prepare_or_load_dataset(run_config, DATA_SOURCES[mode], cache_root)
    PREPARED_DATASETS.append(prepared)
    manifest = json.loads((prepared / 'dataset_manifest.json').read_text(encoding='utf-8'))
    print('Dataset:', prepared)
    print(json.dumps(manifest['stats'], indent=2))


## 5. Start new training runs

Run this cell only when you want **new weights for every configuration**. It creates a separate Drive folder under `MyDrive/LDM_FINAL/runs` for each entry and trains them sequentially with live progress. `last.pt` is for resuming; `best.pt` is chosen by validation loss. The folder also receives the saved config, logs, evaluation history, and final test metrics. Rerunning this cell starts another set of automatically named runs. A custom `name` can be used only once; to continue that run, use Resume in section 7.

Before updates, attention checks verify cache/full prediction agreement, future causality, padding, and disabled streams; results are saved as `model_checks.json`.


In [ ]:
from ldm.experiment_workflow import start_run
assert (
    'PREPARED_CONFIGS' in globals()
    and PREPARED_CONFIGS == tuple(RUN_CONFIGS)
    and globals().get('PREPARED_SOURCE') == globals().get('SELECTED_DATA_SOURCE')
    and globals().get('PREPARED_SOURCE') == current_data_selection()
    and len(PREPARED_DATASETS) == len(RUN_CONFIGS)
), 'Run preparation again after changing RUN_CONFIGS'
STARTED_RUNS = []
for index, (run_config, prepared) in enumerate(zip(RUN_CONFIGS, PREPARED_DATASETS)):
    print(f'Starting config {index + 1}/{len(RUN_CONFIGS)}: {run_config.data.variant.value}')
    run = start_run(run_config, prepared, runs_root)
    STARTED_RUNS.append(run)
    print('Completed run:', run.name)
print('Runs started:', [path.name for path in STARTED_RUNS])


## 6. Select a saved run

This is the only cell where you set `SELECT_RUN_ID`. It lists the Drive runs and selects the last run completed in this notebook when the value is `None`; otherwise, type an existing run folder name. Run this cell before any action in sections 7â€“13. It only reads run metadata and does not load weights. You can skip the training cells when working with an existing run.


In [ ]:
from ldm.experiment_workflow import list_runs, inspect_run

available_runs = list_runs(runs_root)
for item in available_runs:
    print(item['run_id'], item['status'], item.get('last_step'), item.get('best_val_loss'))

SELECT_RUN_ID = None  # Type an existing run ID here; None selects the last completed run from this session.
selected_run_id = SELECT_RUN_ID
if selected_run_id is None and globals().get('STARTED_RUNS'):
    selected_run_id = STARTED_RUNS[-1].name
if selected_run_id not in {item['run_id'] for item in available_runs}:
    raise ValueError('Set SELECT_RUN_ID to one of the listed runs, or train a new run first')
selected_run = runs_root / selected_run_id
print(json.dumps(inspect_run(selected_run), indent=2)[:12000])

def selected_snapshot():
    """Build and validate the selected run's snapshot when an action needs it."""
    import hashlib
    checkpoint = selected_run / 'best.pt'
    if not checkpoint.exists():
        raise FileNotFoundError(f'No best.pt in {selected_run}')
    data_config = inspect_run(selected_run, include_metrics=False)['config']['data']
    variant = data_config['variant']
    decoration = data_config['decoration']
    checkpoint_sha256 = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
    snapshot = project / 'snapshots' / f'{variant}-{decoration}-{checkpoint_sha256[:12]}-{code_commit[:8]}'
    manifest_path = snapshot / 'metadata' / 'release_manifest.json'
    if snapshot.exists() and not manifest_path.exists():
        raise RuntimeError(f'Incomplete snapshot: {snapshot}')
    if not snapshot.exists():
        subprocess.run([sys.executable, '-m', 'ldm.cli', 'snapshot',
                        '--run', str(selected_run), '--output', str(snapshot)], check=True)
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest['source_checkpoint_sha256'] != checkpoint_sha256:
        raise RuntimeError(f'Snapshot checkpoint differs from {checkpoint}')
    subprocess.run([sys.executable, '-m', 'ldm.cli', 'validate', '--snapshot', str(snapshot)], check=True)
    return snapshot, manifest


## 7. Resume the selected run

This cell is **off by default** (`RESUME_NOW = False`). After selecting a run in section 6, enable it to continue from that run's `last.pt`. It reads the saved configuration from Drive. `TARGET_EPOCHS` means total epochs, including completed ones; a lower target is rejected. No metrics, snapshot, or inference cell is needed first.


In [ ]:
from ldm.experiment_workflow import resume_run
RESUME_NOW = False
TARGET_EPOCHS = 10
if RESUME_NOW:
    result = resume_run(selected_run, target_epochs=TARGET_EPOCHS)
    print('Stopped:', result['stop_reason'], 'at step', result['steps'])
else:
    print('Resume skipped. Set RESUME_NOW=True when ready.')


## 8. Inspect metrics and plots

This optional cell reads the selected run's `metrics.json`, prints key test measures, saves a training/validation loss plot in its folder, and writes a comparison CSV under `MyDrive/LDM_FINAL/comparisons`. Validation chose `best.pt`; test metrics describe that checkpoint. Test cycles come from files also represented in training. Other actions do not require this cell.

The comparison cell also saves `test_accuracy.png` and its CSV data. Colors and labels identify groups with matching dataset, split, and metric versions.


In [ ]:
from ldm.experiment_workflow import compare_runs
from ldm.evaluation_ldm import format_percentage, print_evaluation_metrics
from ldm.report_ldm import save_history_plot, save_comparison, save_test_accuracy_comparison
metrics = json.loads((selected_run / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps({key: metrics[key] for key in
                  ('steps', 'best_step', 'best_val_loss', 'test_loss', 'stop_reason')}, indent=2))
print_evaluation_metrics(metrics['test_report'], 'Test')
completed_runs = [runs_root / item['run_id'] for item in list_runs(runs_root)
                  if (runs_root / item['run_id'] / 'metrics.json').exists()]
for row in compare_runs(completed_runs):
    print(row['run_id'], '|', row['status'], '| Test accuracy:', format_percentage(row['test_token_accuracy']))
if inspect_run(selected_run)['config']['evaluation']['save_plots']:
    print('Plot:', save_history_plot(selected_run))
if completed_runs:
    print('Comparison CSV:', save_comparison(completed_runs, storage_root / 'comparisons' / 'runs.csv'))
    print('Test accuracy plot:', save_test_accuracy_comparison(completed_runs, storage_root / 'comparisons' / 'test_accuracy.png'))


## 9. Optional local Hugging Face snapshot check

This cell builds and validates a loadable snapshot from the selected run's `best.pt`. It checks `AutoModel`, `AutoProcessor`, and a five-line round trip. It does not upload anything. Inference, analysis, and release call the same snapshot helper themselves, so you may skip this standalone check.


In [ ]:
snapshot, release_manifest = selected_snapshot()
print('Validated:', snapshot)


## 10. Inference: generate rhythms

This cell uses the selected model's best checkpoint to generate new rhythm cycles. It builds/validates its snapshot itself, so section 9 is optional. Set `INFERENCE_PROMPTS` to complete five-line prompt files, or leave it empty to extract one cycle from `PROMPT_SOURCE_FILE`. With no source path, the example uses cycle 0 of the first alphabetically sorted HF file from the selected run's decoration mode. `INFERENCE_SEEDS` defaults to one seed; add seeds to generate independent alternatives for each prompt. Each attempt logs generation, decoding, writing, and verification, then saves `prompt.derbake`, **generated-only** `generated.derbake`, and `report.json` under `MyDrive/LDM_FINAL/runs/<run-id>/inference/<attempt>/`. The prompt copy records the exact input used. `GENERATED_FILES` holds all output paths from this cell for use in section 11.

`PREVIEW_TOKENS` controls how many accepted generated tokens are printed. Set it to `0` to hide the preview.


In [ ]:
from ldm.inference_ldm import InferenceConfig, run_inference
from ldm.experiment_config import DecorationMode
from ldm.derbake_ldm import RhythmSequence, parse_derbake, write_derbake

snapshot, release_manifest = selected_snapshot()
variant = release_manifest['dataset_variant']
INFERENCE_PROMPTS = []  # Add paths to complete five-line prompt files, or use the example below.
PROMPT_SOURCE_FILE = None  # Optional path to a five-line source file.
PROMPT_CYCLE_INDEX = 0  # Zero-based cycle within the source file.
if not INFERENCE_PROMPTS:
    if PROMPT_SOURCE_FILE is None:
        mode = DecorationMode(release_manifest['dataset_decoration'])
        sources = globals().get('DATA_SOURCES', {})
        if mode not in sources or not sources[mode].is_dir():
            raise RuntimeError(f'Set a prompt path or load the {mode.value} HF source for this run')
        PROMPT_SOURCE_FILE = sorted(sources[mode].glob('*.derbake'))[0]
    PROMPT_SOURCE_FILE = Path(PROMPT_SOURCE_FILE)
    source_rhythm = parse_derbake(PROMPT_SOURCE_FILE).rhythm
    assert 0 <= PROMPT_CYCLE_INDEX < len(source_rhythm.cycles)
    prompt_dir = selected_run / 'inference' / 'prompts'
    prompt_dir.mkdir(parents=True, exist_ok=True)
    prompt_file = prompt_dir / f'{PROMPT_SOURCE_FILE.stem}_cycle{PROMPT_CYCLE_INDEX + 1}.derbake'
    write_derbake(RhythmSequence((source_rhythm.cycles[PROMPT_CYCLE_INDEX],)), prompt_file)
    INFERENCE_PROMPTS = [prompt_file]

INFERENCE_SEEDS = [1337]  # Add seeds for independent alternatives from each prompt.
PREVIEW_TOKENS = 120  # Number of generated tokens to print; 0 disables the preview.
GENERATED_FILES = []
INFERENCE_RUN_ID = selected_run.name
print('Selected model:', variant, release_manifest['dataset_decoration'])
for prompt in INFERENCE_PROMPTS:
    for seed in INFERENCE_SEEDS:
        report = run_inference(snapshot, prompt, selected_run / 'inference',
                               InferenceConfig(num_cycles=1, seed=seed, max_new_tokens=2048,
                                               preview_tokens=PREVIEW_TOKENS))
        GENERATED_FILES.append(Path(report['output']))


## 11. Optional audio and model analyses

Each cell below is independent. Audio renders an existing `.derbake` using WAV samples; static embeddings and model checks load the selected snapshot when enabled.

### 11.1 Render the prompt and generated rhythm to WAV

Set `RENDER_AUDIO = True` and provide WAV samples in `LDM_FINAL/sounds/normal_hits/{doums,taks,tiks,pa2s}` and `LDM_FINAL/sounds/decorated_hits/{doums,taks,pa2s}`. Set `GENERATED_FILE` to an inference `generated.derbake`, or leave it `None` to use the most recent entry in section 10's `GENERATED_FILES` for this run. Its folder must also contain `prompt.derbake`. The cell renders both files and saves `prompt.wav` and `generated.wav` in that same folder. `AUDIO_SEED` controls sample choice.


In [ ]:
from ldm.dotderbake import render_wav

SOUND_ROOT = Path('/content/drive/MyDrive/LDM_FINAL/sounds')
SAMPLE_DIRS = {'normal_hits': SOUND_ROOT / 'normal_hits', 'decorated_hits': SOUND_ROOT / 'decorated_hits'}
RENDER_AUDIO = False  # Set True after adding the sound folders on Drive.
GENERATED_FILE = None  # Optional generated.derbake path; prompt.derbake must be beside it.
AUDIO_SAMPLE_RATE = 48000
AUDIO_SEED = 1337
SKELETON_GAIN = 3.0  # Original skeleton volume; generated files have no skeleton events.
if RENDER_AUDIO:
    audio_source = GENERATED_FILE
    if audio_source is None and globals().get('INFERENCE_RUN_ID') == selected_run.name and globals().get('GENERATED_FILES'):
        audio_source = GENERATED_FILES[-1]
    if audio_source is None:
        raise ValueError('Set GENERATED_FILE or run inference for this selected run')
    audio_source = Path(audio_source)
    audio_files = (audio_source.parent / 'prompt.derbake', audio_source)
    for rhythm_file in audio_files:
        if not rhythm_file.is_file():
            raise FileNotFoundError(f'Missing inference rhythm: {rhythm_file}')
    for rhythm_file in audio_files:
        print(render_wav(rhythm_file, SAMPLE_DIRS, rhythm_file.with_suffix('.wav'),
                         sample_rate=AUDIO_SAMPLE_RATE, seed=AUDIO_SEED,
                         skeleton_gain=SKELETON_GAIN))
else:
    print('Audio skipped: set RENDER_AUDIO = True to use the normal and decorated sound folders.')


### 11.2 Static embeddings and optional contextual PCA

Set `RUN_STATIC_EMBEDDINGS = True` to compare the selected representation modes. `TOKEN_ONLY` uses token vectors; `BASE_INPUT` includes the actual type addition or concatenation projection; `OUTPUT_HEAD` uses prediction vectors. Each mode saves neighbor CSVs, subdivision summaries, and purity/cosine statistics under the selected run's `analysis/embeddings` folder. Subdivision statistics are marked unavailable for `no_subd`; HIT neighbors remain available. `RUN_CONTEXTUAL_ANALYSIS` independently enables the existing contextual PCA CSV.

With tied weights, token-only and output-head vectors are the same learned table; base-input vectors can still differ because of type embeddings.


In [ ]:
from ldm.embedding_analysis import EmbeddingRepresentation, analyze_static_embeddings
from ldm.inference_ldm import contextual_projection

RUN_STATIC_EMBEDDINGS = False
EMBEDDING_REPRESENTATIONS = [
    EmbeddingRepresentation.TOKEN_ONLY,
    EmbeddingRepresentation.BASE_INPUT,
    EmbeddingRepresentation.OUTPUT_HEAD,
]
NEIGHBOR_TOP_K = 8
RUN_CONTEXTUAL_ANALYSIS = False
if RUN_STATIC_EMBEDDINGS or RUN_CONTEXTUAL_ANALYSIS:
    snapshot, release_manifest = selected_snapshot()
if RUN_STATIC_EMBEDDINGS:
    result = analyze_static_embeddings(snapshot, selected_run / 'analysis' / 'embeddings',
                                       representations=EMBEDDING_REPRESENTATIONS, top_k=NEIGHBOR_TOP_K)
    print(json.dumps(result, indent=2))
if RUN_CONTEXTUAL_ANALYSIS:
    selected_dataset = Path(inspect_run(selected_run)['dataset_path'])
    print(contextual_projection(snapshot, selected_dataset, selected_run / 'analysis'))


### 11.3 Model checks and CUDA profiling

`RUN_MODEL_CHECKS` checks the selected model's full/prefill/cached predictions, partial windows, future causality, padding, and disabled-stream gates. These checks also run before training. `RUN_CUDA_PROFILE` measures forward/gradient time, peak allocated memory, padding utilization, and deterministic cached decoding on a GPU, without optimizer updates. Reports are saved under `analysis/validation` and `analysis/profiling`. For a baseline comparison, profile a global run and a structured run on the same hardware, data, and numeric settings, then set `BASELINE_PROFILE` to the global run's `cuda_profile.json`. The comparison rejects mismatched settings and reports the old speed/memory acceptance gate. GPU profiling is optional and off by default.


In [ ]:
RUN_MODEL_CHECKS = False
RUN_CUDA_PROFILE = False
CHECK_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PROFILE_BATCH_SIZE = 1
PROFILE_WARMUP_STEPS = 3
PROFILE_MEASURED_STEPS = 10
PROFILE_AMP = True
PROFILE_ALLOW_TF32 = True
BASELINE_PROFILE = None  # Path to a matching global run's cuda_profile.json.
if RUN_MODEL_CHECKS or RUN_CUDA_PROFILE:
    from ldm.model_checks import run_model_checks, profile_cuda, compare_profiles
    snapshot, release_manifest = selected_snapshot()
if RUN_MODEL_CHECKS:
    print(json.dumps(run_model_checks(snapshot, selected_run / 'analysis' / 'validation',
                                      device=CHECK_DEVICE), indent=2))
if RUN_CUDA_PROFILE:
    selected_dataset = Path(inspect_run(selected_run)['dataset_path'])
    profile = profile_cuda(snapshot, selected_dataset, selected_run / 'analysis' / 'profiling',
                           batch_size=PROFILE_BATCH_SIZE, warmup_steps=PROFILE_WARMUP_STEPS,
                           measured_steps=PROFILE_MEASURED_STEPS, amp=PROFILE_AMP,
                           allow_tf32=PROFILE_ALLOW_TF32)
    print(json.dumps(profile, indent=2))
    if BASELINE_PROFILE is not None:
        baseline = json.loads(Path(BASELINE_PROFILE).read_text(encoding='utf-8'))
        comparison = compare_profiles(baseline, profile)
        output = selected_run / 'analysis' / 'profiling' / 'baseline_comparison.json'
        output.write_text(json.dumps(comparison, indent=2) + '\n', encoding='utf-8')
        print(json.dumps(comparison, indent=2))


## 12. Optional Hugging Face release

This cell builds/validates the selected snapshot and reads the selected run's final metrics itself. It shows files that a release would replace or remove in the matching subdivision model repo. Normal and decorated experiments currently share that variant's release destination; publish only the selected experiment you want there. `PUBLISH_TO_HF = False` means **nothing is uploaded**. Set it to `True` only for the run you want to publish; training checkpoints stay on Drive. You do not need to run metrics, snapshot, or inference cells first.


In [ ]:
from huggingface_hub import HfApi

snapshot, release_manifest = selected_snapshot()
variant = release_manifest['dataset_variant']
metrics = json.loads((selected_run / 'metrics.json').read_text(encoding='utf-8'))
repo_id = {'with_subd': 'Appodoom/ldm-with-subd', 'no_subd': 'Appodoom/ldm-no-subd'}[variant]
print('Selected dataset:', variant, release_manifest['dataset_decoration'])
api = HfApi()
remote_info = api.repo_info(repo_id=repo_id, repo_type='model')
remote_files = {item.rfilename for item in remote_info.siblings}
local_files = {path.relative_to(snapshot).as_posix() for path in snapshot.rglob('*') if path.is_file()}
stale_files = sorted(remote_files - local_files - {'.gitattributes'})
print('Remote files to remove on release:', stale_files)
PUBLISH_TO_HF = False
published_commit = None
published_run_id = None
if PUBLISH_TO_HF:
    if metrics['stop_reason'] not in {'completed', 'early_stopped'}:
        raise ValueError('Selected run has not completed training')
    uploaded = api.upload_folder(repo_id=repo_id, repo_type='model', folder_path=str(snapshot),
                                 commit_message=f'Release {variant} model',
                                 parent_commit=remote_info.sha, delete_patterns=stale_files)
    published_commit = uploaded.oid
    published_run_id = selected_run.name
    print(f'https://huggingface.co/{repo_id}/tree/{published_commit}')
else:
    print('Upload skipped.')


## 13. Verify a Hugging Face commit

Set `VERIFY_COMMIT` to a model repo commit SHA, or leave it `None` to use the commit just uploaded for this selected run in section 12. This cell downloads that exact revision, checks isolated loading and a five-line round trip, and saves a release record under `MyDrive/LDM_FINAL/releases`. You can verify an earlier upload without rerunning the upload cell. If neither commit is available, it skips.


In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoProcessor
import hashlib

VERIFY_COMMIT = None  # Or paste an existing HF model commit SHA here.
if VERIFY_COMMIT is None and globals().get('published_run_id') == selected_run.name:
    VERIFY_COMMIT = globals().get('published_commit')
if VERIFY_COMMIT is None:
    print('Remote validation skipped: set VERIFY_COMMIT or publish this run first.')
else:
    run_record = inspect_run(selected_run, include_metrics=False)
    variant = run_record['config']['data']['variant']
    repo_id = {'with_subd': 'Appodoom/ldm-with-subd', 'no_subd': 'Appodoom/ldm-no-subd'}[variant]
    metrics = json.loads((selected_run / 'metrics.json').read_text(encoding='utf-8'))
    checkpoint_sha256 = hashlib.sha256((selected_run / 'best.pt').read_bytes()).hexdigest()
    remote_snapshot = Path(snapshot_download(repo_id=repo_id, repo_type='model',
        revision=VERIFY_COMMIT, local_dir=project / 'remote_validation' / variant / VERIFY_COMMIT))
    subprocess.run([sys.executable, '-m', 'ldm.cli', 'validate', '--snapshot', str(remote_snapshot)], check=True)
    remote_manifest = json.loads((remote_snapshot / 'metadata' / 'release_manifest.json').read_text(encoding='utf-8'))
    if remote_manifest['source_checkpoint_sha256'] != checkpoint_sha256:
        raise ValueError('HF revision does not match the selected run checkpoint')
    if remote_manifest['dataset_variant'] != variant:
        raise ValueError('HF revision has the wrong dataset variant')
    if remote_manifest['dataset_decoration'] != run_record['config']['data']['decoration']:
        raise ValueError('HF revision has the wrong decoration mode')
    remote_model = AutoModel.from_pretrained(repo_id, revision=VERIFY_COMMIT, trust_remote_code=True).eval()
    remote_processor = AutoProcessor.from_pretrained(repo_id, revision=VERIFY_COMMIT, trust_remote_code=True)
    assert remote_model.config.vocab_hash == remote_processor.tokenizer.atomic.hash
    release_record = {'repo_id': repo_id, 'revision': VERIFY_COMMIT, 'run_id': selected_run.name,
        'code_commit': run_record.get('code_commit'), 'checkpoint_sha256': checkpoint_sha256,
        'dataset_fingerprint': metrics['dataset_fingerprint'],
        'decoration': run_record['config']['data']['decoration'],
        'metric_version': metrics['test_report']['metric_version']}
    record_path = storage_root / 'releases' / f'{variant}-{VERIFY_COMMIT[:12]}.json'
    record_path.parent.mkdir(parents=True, exist_ok=True)
    record_path.write_text(json.dumps(release_record, indent=2) + '\n', encoding='utf-8')
    print('Verified:', VERIFY_COMMIT, '| Record:', record_path)
